# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library, referencing dataset entities by their `@id` as per the Croissant specification.

### Dataset Source
This dataset's Croissant schema is available at the following URL:
- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant JSON-LD Schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Let's list the available record sets, fields, and columns, referencing their `@id`s as specified in the Croissant schema.

In [ ]:
# List all record sets by @id
print('Available record sets:')
record_sets = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
for rs in record_sets:
    print(f"- {rs}")

# For demonstration, print fields of the first record set, referenced by @id
if record_sets:
    record_set_id = record_sets[0]
    print(f"\nFields in record set '@id': {record_set_id}")
    # Retrieve the full record set dict for the given @id
    recset = next(
        item for item in metadata.to_json()['recordSet'] if item['@id'] == record_set_id
    )
    fields = recset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for fld in fields:
        if isinstance(fld, dict):
            print(f"  - {fld['@id']}")
        else:
            print(f"  - {fld}")
else:
    print('No record sets found in metadata.')

## 3. Data Extraction
Let's load data from the record sets. We'll extract all record sets defined above (by their `@id`), reading each into a pandas DataFrame for analysis.
Use the record set and field `@id`s explored in the previous cell.

In [ ]:
# Extract data from all record sets into DataFrames, indexed by @id
dataframes = {}

for recset_id in record_sets:
    print(f"Loading records for record set: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    dataframes[recset_id] = pd.DataFrame(records)
    print(f"  -> Loaded {len(dataframes[recset_id])} records")

if record_sets:
    print(f"\nColumns in record set {record_sets[0]}: {list(dataframes[record_sets[0]].columns)}\n")
    display(dataframes[record_sets[0]].head())

## 4. Exploratory Data Analysis (EDA)

Now, let's perform EDA on the data.
- We'll select a numeric field from the available columns using field `@id` (as revealed above).
- The workflow demonstrates: filtering, normalization, and grouping (if categorical fields are found).

In [ ]:
# Identify a record set for EDA
# In this example, we use the first record set if present
if not record_sets:
    raise RuntimeError('No record sets available for analysis.')

record_set_id = record_sets[0]
df = dataframes[record_set_id]

# Print field (column) list to help pick a numeric field (replace with actual @id as needed)
print(f"Fields (columns) available in record set {record_set_id}:")
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

# Let's heuristically select the first column that looks numeric for this example
import numpy as np
numeric_field_id = None
for col in df.columns:
    if np.issubdtype(df[col].dropna().apply(lambda x: pd.to_numeric(x, errors='coerce')).dtype, np.number):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # If no numeric fields found, skip EDA
    raise RuntimeError('No numeric fields found for EDA!')

print(f"\nNumeric field chosen for analysis: {numeric_field_id}\n")

# Filter: keep only those with value > threshold
threshold = df[numeric_field_id].dropna().apply(lambda x: pd.to_numeric(x, errors='coerce')).median()
filtered_df = df[df[numeric_field_id].apply(lambda x: pd.to_numeric(x, errors='coerce')) > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold} ({len(filtered_df)} records):")
display(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id].apply(lambda x: pd.to_numeric(x, errors='coerce')) -
    filtered_df[numeric_field_id].apply(lambda x: pd.to_numeric(x, errors='coerce')).mean()
) / filtered_df[numeric_field_id].apply(lambda x: pd.to_numeric(x, errors='coerce')).std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If any field can be treated as a categorical/grouping variable, demonstrate grouping
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and df[col].nunique() < 10:
        group_field_id = col
        break
if group_field_id:
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean').reset_index()
    )
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric variable, and if a grouping variable exists, show the mean by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of normalized numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True, bins=15)
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id} (normalized)")
plt.ylabel("Count")
plt.show()

# If grouping variable available, plot barplot of mean values
if group_field_id:
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field_id, y="mean", data=grouped_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion

- We demonstrated how to load a Croissant-compliant clinical dataset and referenced all dataset entities by their `@id`.
- Using `mlcroissant` and pandas, we explored available record sets and fields, and extracted tabular data for EDA.
- We performed normalization and simple groupings, and visualized distributions using standard Python visualization libraries.

For further investigation, we recommend referencing this notebook template and the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) for more advanced schema-aware data processing pipelines.